# Lekcija 34: Konvolucijske neuronske mreže (CNN)

Slojevi iz lekcija 31–33 svaki ulaz tretiraju kao jednodimenzijski vektor, što za slike stvara dva odvojena problema. Prvo, svaki piksel dobiva vlastitu neovisnu težinu, bez informacije da su susjedni pikseli *blizu* jedan drugome — odbacuje se cijela prostorna struktura slike. Drugo, budući da svaki položaj dobiva zasebne težine, potpuno povezani sloj ne može ponovno koristiti već naučeni filtar: mora neovisno i iznova učiti kako rub izgleda na svakom pojedinom položaju. **Konvolucija** &mdash; već ručno izgrađena u lekciji 10 &mdash; rješava oba problema odjednom ugrađivanjem **induktivne pristranosti**: svaki izlaz ograničava na malo lokalno područje i dijeli težine tog područja među svim položajima.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

## Konvolucijski sloj je ograničena i dijeljena projekcija

Prisjetite se središnje ideje lekcije 30: $y = w^\top x + b$, projekcija. Konvolucijski sloj računa *upravo to*, ali za $x$ gleda samo malo lokalno područje (recimo $3\times3$) umjesto cijele slike &mdash; i, presudno, ponovno koristi *isti* $w$ na svakom položaju područja. Dvije posljedice:

- **Mnogo manje parametara.** Filtar $3\times3$ nad jednim ulaznim kanalom ima 9 težina, neovisno o veličini slike &mdash; potpuno povezani sloj nad slikom $16\times16$ treba 256 težina *po izlaznoj jedinici*.
- **Ekvivarijantnost na translaciju.** Budući da isti filtar klizi svugdje, značajka naučena na jednom položaju automatski se i besplatno detektira i na svakom drugom. *Ekvivarijantno* znači da se izlaz pomiče zajedno s ulazom, umjesto da ostaje fiksan &mdash; pomaknite sliku za 2 piksela i detekcije na karti značajki također se pomiču za točno 2 piksela. To je zasebno, slabije svojstvo od *invarijantnosti* (nepromijenjen izlaz unatoč pomaknutom ulazu), koje sažimanje djelomično dodaje kasnije u lekciji.

## Jezgra konvolucijskog sloja je 3D

Još nešto: u kontekstu konvolucijske neuronske mreže (CNN), „filtar $3\times3$” zapravo je $3\times3\times C_{in}$, a ne samo $3\times3$. Jezgre iz lekcije 10 bile su ravne 2D mreže jer su klizile preko jednog kanala sivih tonova. Filtar konvolucijskog sloja mora obuhvatiti *svaki* ulazni kanal odjednom &mdash; filtar $3\times3$ na trokanalnoj RGB slici ima $3\times3\times3=27$ težina, po jedan puni presjek dubine za svaki ulazni kanal, zbrojenih u jednu izlaznu vrijednost na svakom položaju. Posložite $C_{out}$ takvih filtara $3\times3\times C_{in}$ jedan uz drugi (po jedan za svaki izlazni kanal) i cijeli tenzor težina je 4D: `(C_out, C_in, 3, 3)` &mdash; upravo oblik koji očekuje `conv2d_forward` u nastavku.

## Conv2d: prolaz unaprijed i unatrag, od početka

To je isto računanje kliznim prozorom kao konvolucija iz lekcije 10, samo što je jezgra sada skup *težina koje se mogu učiti* umjesto fiksne, ručno oblikovane jezgre &mdash; što znači da joj treba i prolaz unatrag.

In [ ]:
def conv2d_forward(img, kernel, bias):
    N, C, H, W = img.shape
    OC, IC, KH, KW = kernel.shape
    OH, OW = H - KH + 1, W - KW + 1
    out = np.zeros((N, OC, OH, OW))
    for n in range(N):
        for oc in range(OC):
            for i in range(OH):
                for j in range(OW):
                    patch = img[n, :, i:i + KH, j:j + KW]
                    out[n, oc, i, j] = np.sum(patch * kernel[oc]) + bias[oc]
    return out

def conv2d_backward(img, kernel, grad_out):
    N, C, H, W = img.shape
    OC, IC, KH, KW = kernel.shape
    OH, OW = grad_out.shape[2], grad_out.shape[3]
    grad_kernel = np.zeros_like(kernel)
    grad_img = np.zeros_like(img)
    grad_bias = grad_out.sum(axis=(0, 2, 3))
    for n in range(N):
        for oc in range(OC):
            for i in range(OH):
                for j in range(OW):
                    patch = img[n, :, i:i + KH, j:j + KW]
                    grad_kernel[oc] += grad_out[n, oc, i, j] * patch
                    grad_img[n, :, i:i + KH, j:j + KW] += grad_out[n, oc, i, j] * kernel[oc]
    return grad_img, grad_kernel, grad_bias

### Provjera ispravnosti pomoću PyTorcha

In [ ]:
rng = np.random.default_rng(0)
img = rng.normal(size=(1, 3, 8, 8))          # 3 input channels, e.g. RGB
kernel = rng.normal(size=(1, 3, 3, 3)) * 0.5  # ONE output filter, spanning all 3 input channels
bias = np.array([0.1])

out = conv2d_forward(img, kernel, bias)

img_t = torch.tensor(img, requires_grad=True)
kernel_t = torch.tensor(kernel, requires_grad=True)
bias_t = torch.tensor(bias, requires_grad=True)
out_t = F.conv2d(img_t, kernel_t, bias_t)
print(f'forward max diff: {np.abs(out - out_t.detach().numpy()).max():.2e}')

grad_out = rng.normal(size=out.shape)
grad_img, grad_kernel, grad_bias = conv2d_backward(img, kernel, grad_out)
out_t.backward(torch.tensor(grad_out))

print(f'grad_img max diff:    {np.abs(grad_img - img_t.grad.numpy()).max():.2e}')
print(f'grad_kernel max diff: {np.abs(grad_kernel - kernel_t.grad.numpy()).max():.2e}')
print(f'grad_bias max diff:   {np.abs(grad_bias - bias_t.grad.numpy()).max():.2e}')

## Filtri koje već poznajete: detektori rubova iz lekcije 12

Izlaz konvolucijskog sloja prije treniranja nema smisla; poanta je da će gradijentni spust *pronaći* korisne filtre. Kako bismo vidjeli kako izlaz korisnog filtra već izgleda, primijenimo filtar koji smo ručno oblikovali u lekciji 12: Sobelovu jezgru rubova. Pokazuje se da filtri prvog sloja treniranih CNN-ova često konvergiraju prema vizualno sličnim oblicima — detektorima orijentiranih rubova i područja — neovisno o zadatku za koji je mreža trenirana.

In [ ]:
test_img = np.zeros((1, 1, 40, 40))
test_img[0, 0, 10:30, 10:30] = 1.0

sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float64).reshape(1, 1, 3, 3)
feature_map = conv2d_forward(test_img, sobel_x, np.zeros(1))

fig, axes = plt.subplots(1, 2, figsize=(6, 3.5))
axes[0].imshow(test_img[0, 0], cmap='gray')
axes[0].set_title('Input')
axes[1].imshow(feature_map[0, 0], cmap='gray')
axes[1].set_title('Feature map\n(Sobel filter, as a conv layer)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## Sažimanje: smanjivanje rezolucije sa svrhom

**Sažimanje maksimumom (max pooling)** pomiče prozor preko karte značajki i zadržava samo najveću vrijednost u svakom prozoru, smanjujući prostornu veličinu (poput piramida iz lekcije 11), uz dodavanje male količine lokalne invarijantnosti na translaciju &mdash; značajka detektirana na malo različitim položajima unutar jednog prozora sažimanja i dalje daje isti izlaz.

In [ ]:
def maxpool2d(x, size=2):
    N, C, H, W = x.shape
    OH, OW = H // size, W // size
    out = np.zeros((N, C, OH, OW))
    for i in range(OH):
        for j in range(OW):
            patch = x[:, :, i * size:i * size + size, j * size:j * size + size]
            out[:, :, i, j] = patch.max(axis=(2, 3))
    return out

pool_test = rng.normal(size=(1, 1, 8, 8))
mine = maxpool2d(pool_test)
torch_result = F.max_pool2d(torch.tensor(pool_test), 2)
print(f'max pooling max diff vs torch: {np.abs(mine - torch_result.numpy()).max():.2e}')

## Povezivanje: naizmjenična konvolucija i sažimanje

Stvarni CNN ne sastoji se samo od konvolucije ili samo od sažimanja &mdash; oni se izmjenjuju: konvolucija, sažimanje, konvolucija, sažimanje i tako dalje. Svaki korak sažimanja približno *prepolovljuje* prostornu veličinu ($H$ i $W$), a praktično je pravilo približno *udvostručiti* broj kanala u sljedećem konvolucijskom sloju. To nije slučajno: sažimanje odbacuje prostornu rezoluciju, pa udvostručavanje broja kanala odmah nakon njega održava ukupnu količinu informacija ($C \times H \times W$) približno stalnom između faza, iako prikaz mijenja „malo detalja na mnogim položajima” za „više detalja na manje položaja”. `CNNClassifier` izgrađen u nastavku slijedi taj obrazac: prvi konvolucijski sloj s $1 \rightarrow 8$ kanala pri punoj rezoluciji $16\times16$; sažimanje koje je prepolovljuje na $8\times8$; zatim drugi konvolucijski sloj koji ponovno udvostručuje kanale, $8 \rightarrow 16$.

## Zašto je ekvivarijantnost na translaciju važna: test generalizacije

Izgradimo mali klasifikacijski zadatak: slika $16\times16$ sadrži znak plus ili krug na *nasumičnom položaju*. Treniramo na oblicima smještenima samo blizu središta, a testiramo na oblicima smještenima samo blizu kutova &mdash; položajima koje model nikada nije vidio pri treniranju. Mreža koja je doista naučila „kako plus izgleda”, a ne „koji su pikseli obično uključeni za plus na ovim konkretnim položajima za treniranje”, ne bi trebala imati poteškoća s time.

In [ ]:
def make_image(shape_type, cx, cy, size=16):
    img = np.zeros((size, size), dtype=np.float32)
    if shape_type == 'plus':
        img[cy - 1:cy + 2, cx - 3:cx + 4] = 1.0
        img[cy - 3:cy + 4, cx - 1:cx + 2] = 1.0
    else:
        yy, xx = np.mgrid[0:size, 0:size]
        img[((xx - cx) ** 2 + (yy - cy) ** 2) <= 9] = 1.0
    return img

def make_dataset(rng_local, n, position_range):
    imgs, labels = [], []
    for _ in range(n):
        shape_type = rng_local.choice(['plus', 'circle'])
        cx, cy = rng_local.integers(*position_range), rng_local.integers(*position_range)
        imgs.append(make_image(shape_type, cx, cy))
        labels.append(0.0 if shape_type == 'plus' else 1.0)
    return np.array(imgs, dtype=np.float32), np.array(labels, dtype=np.float32)

data_rng = np.random.default_rng(1)
X_train, y_train = make_dataset(data_rng, 300, (5, 11))   # near center
X_test, y_test = make_dataset(data_rng, 150, (3, 5))       # near a corner -- never seen in training

fig, axes = plt.subplots(1, 4, figsize=(9, 2.5))
for ax, im, title in zip(axes, [X_train[0], X_train[1], X_test[0], X_test[1]],
                          ['train example', 'train example', 'test example\n(unseen position)', 'test example\n(unseen position)']):
    ax.imshow(im, cmap='gray')
    ax.set_title(title, fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
class MLPClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(), nn.Linear(16 * 16, 32), nn.ReLU(), nn.Linear(32, 1))

    def forward(self, x):
        return self.net(x).squeeze(-1)

class CNNClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 8, 5, padding=2), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, 5, padding=2), nn.ReLU(),
            nn.AdaptiveMaxPool2d(1),  # global max pool: collapses ALL spatial position info
        )
        self.fc = nn.Linear(16, 1)

    def forward(self, x):
        return self.fc(self.conv(x).flatten(1)).squeeze(-1)

def train_and_eval(model_cls, Xtr, ytr, Xte, yte, seed, epochs=400, lr=0.01):
    torch.manual_seed(seed)  # seed BEFORE constructing the model, so init is actually reproducible
    model = model_cls()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(epochs):
        opt.zero_grad()
        loss = F.binary_cross_entropy_with_logits(model(Xtr), ytr)
        loss.backward()
        opt.step()
    with torch.no_grad():
        train_acc = ((model(Xtr) > 0).float() == ytr).float().mean().item()
        test_acc = ((model(Xte) > 0).float() == yte).float().mean().item()
    return train_acc, test_acc

Xtr_t = torch.tensor(X_train).unsqueeze(1)
ytr_t = torch.tensor(y_train)
Xte_t = torch.tensor(X_test).unsqueeze(1)
yte_t = torch.tensor(y_test)

mlp_train_acc, mlp_test_acc = train_and_eval(MLPClassifier, Xtr_t, ytr_t, Xte_t, yte_t, seed=0)
cnn_train_acc, cnn_test_acc = train_and_eval(CNNClassifier, Xtr_t, ytr_t, Xte_t, yte_t, seed=0)

print(f'{"model":>6} {"train acc":>10} {"test acc (unseen positions)":>30}')
print(f'{"MLP":>6} {mlp_train_acc:>10.1%} {mlp_test_acc:>30.1%}')
print(f'{"CNN":>6} {cnn_train_acc:>10.1%} {cnn_test_acc:>30.1%}')

Oba modela savršeno prilagođavaju podatke za treniranje. Na položajima koje nijedan nije vidio, MLP koji pretvara sliku u vektor jedva nadmašuje bacanje novčića &mdash; zapamtio je *koji su pikseli* obično uključeni za svaku klasu na položajima za treniranje, a to znanje nije prenosivo. CNN, čije globalno sažimanje maksimumom prisiljava završnu odluku da ovisi samo o tome *koji su se filtri negdje aktivirali*, a ne *gdje*, generalizira na nove položaje bez ikakvog gubitka točnosti.

### Zadaci

1. Zamijenite `nn.AdaptiveMaxPool2d(1)` u `CNNClassifier` s `nn.Flatten()` izravno nad konvolucijskim značajkama (uklanjajući globalno sažimanje kako bi prostorni položaj ostao očuvan sve do završnog linearnog sloja). Ponovno trenirajte i evaluirajte na testnom skupu s neviđenim položajima. Ostaje li CNN-ova prednost u generalizaciji?
2. Povećajte razmak između raspona položaja za treniranje i testiranje (npr. trenirajte na `(5, 11)`, testirajte na `(0, 3)`, uz sam rub slike gdje su oblici odrezani). Održava li se točnost CNN-a ili opada &mdash; a ako opada, je li to neuspjeh invarijantnosti na translaciju ili nešto sasvim drugo (razmislite što se uz rub događa s *izgledom* oblika, a ne samo njegovim položajem)?
3. Gornji `conv2d_forward` ne koristi popunjavanje (`OH = H - KH + 1`), pa se izlaz smanjuje sa svakim slojem. Izmijenite ga tako da podržava popunjavanje nulama (prije pomicanja prozora dodajte `KH//2` na svaku stranu ulaza, u skladu s PyTorchovim argumentom `padding=`) i potvrdite da je izlazna veličina jednaka ulaznoj, uz provjeru pomoću `F.conv2d(..., padding=1)`.